In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
import os 
from getpass import getpass
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [13]:
## deterministic approach of guardrailing 

import re

def deterministic_guardrail(text:str)->bool:
    """Returns True if content is blocked"""
    banned_keywords=["hack","exploit","malware","bomb","attack","phishing"]
    return any(kw in text.lower() for kw in banned_keywords)

text_inputs=[
    "How to hack into a computer?",
    "What is the capital of France?",
    "Explain how malware spreads."]

print("<< Deterministic Guardrail Results: >>")

for inp in text_inputs:
    blocked=deterministic_guardrail(inp)
    status="Blocked" if blocked else "Allowed"
    print(f"Input: '{inp}' - {status}")

<< Deterministic Guardrail Results: >>
Input: 'How to hack into a computer?' - Blocked
Input: 'What is the capital of France?' - Allowed
Input: 'Explain how malware spreads.' - Blocked


In [ ]:
## model based approach of guardrailing

from langchain_groq import ChatGroq

model = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

text_inputs=[
    "How to hack into a computer?",
    "What is the capital of France?",
    "Explain how to create a malware."]

def model_based_guardrail(text:str)->str:
    """Use a LLM to evaluate content safety Returns SAFE or UNSAFE"""
    
    prompt=f""" is the following user input safe to process ? reply with only 'safe' or 'unsafe' .
    Input: {text}"""
    result=model.invoke([{"role":"user","content":prompt}])
    return result.content.strip()

print("\n<< Model-Based Guardrail Results: >>")
for inp in text_inputs:
    result=model_based_guardrail(inp)
    status="UNSAFE" if "unsafe" in result.lower() else "SAFE"
    print(f"Input: '{inp}' - {result.upper()}")


<< Model-Based Guardrail Results: >>
Input: 'How to hack into a computer?' - UNSAFE
Input: 'What is the capital of France?' - SAFE
Input: 'Explain how to create a malware.' - UNSAFE


In [19]:
# PII -> personal identifiable information {email,credit_card,ip,mac_address,url}
# strategy masking,hashing,blocking & redacting 

from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_core.tools import tool
from langchain_groq import ChatGroq

In [21]:
@tool
def customer_lookup(query:str)->str:
    """look up for customer Information """
    return f"Customer info for query: {query}"

agent=create_agent(
    model=ChatGroq(model="llama-3.3-70b-versatile",temperature=0),
    tools=[customer_lookup],
    middleware=
[       # redact email before sending to model ie abc@gmail.com -> [REDACTED_EMAIL]
        PIIMiddleware(
        "email",
        strategy="redact",
        apply_to_input=True,
    ),
    
    # mask credit card numbers before sending to model ie 1234-5678-9012-3456 -> 1234-****-**
    PIIMiddleware(
        "credit_card",
        strategy="mask",
        apply_to_input=True,
    ),
    
    # blocks API key raise-error if detected in input
    PIIMiddleware(
        "api_key",
        detector=r"sk-[a-zA-Z0-9]{32,}", # regular expression a-z,A-Z,0-9 with length of 32 starting with sk-
        strategy="block",
        apply_to_input=True,
    ),
],
    
)

print("Agent with PII middleware created successfully.")

Agent with PII middleware created successfully.


In [28]:
# test PII Redaction
result=agent.invoke({
    "messages":[{
        "role":"user",
        "content":"my email is abc@gmail.com and my credit_card number is 4242-4242-4242-4242 . Can you help me with my account ?"
    }]
})

print("*" * 50)
print("Result:")
print(result["messages"][-1].content)

**************************************************
Result:
I've located your account using the email address you provided. For security reasons, I couldn't use your credit card number to search for your account. If you have any other questions or concerns about your account, please let me know, and I'll be happy to help.


In [29]:
result

{'messages': [HumanMessage(content='my email is [REDACTED_EMAIL] and my credit_card number is ****-****-****-4242 . Can you help me with my account ?', additional_kwargs={}, response_metadata={}, id='8a6695a1-b9b7-485c-91da-2d492ec3b814'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'wevdeaw6j', 'function': {'arguments': '{"query":"[REDACTED_EMAIL]"}', 'name': 'customer_lookup'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 243, 'total_tokens': 262, 'completion_time': 0.048384332, 'completion_tokens_details': None, 'prompt_time': 0.013177461, 'prompt_tokens_details': None, 'queue_time': 0.161122178, 'total_time': 0.061561793}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019eb297-f909-7962-a939-269215b0a2aa-0', tool_calls=[{'name': 'customer_lookup', 'args': {'q

In [30]:
# Testing the agent with Api key in input to see if it gets blocked by the middleware
try:
    result=agent.invoke({
        "messages":[{
            "role":"user",
            "content":"my api key is sk-1234567890abcdef1234567890abcdef . Can you help me with my account ?"
        }]  
    })
except Exception as e:
    print("*" * 50)
    print("Error:")
    print(e)

**************************************************
Error:
Detected 1 instance(s) of api_key in text content
